# WPTG 2026-2027 – Wybrane Problemy Teorii Grafów
### Projekt: System transformacji grafowych (Graph Grammars)
**Prowadzący:** dr hab. Adam Sędziwy, prof. AGH

Notatnik demonstruje działanie modularnego pakietu `wptg` na oficjalnych danych testowych ze specyfikacji:
- **Graf testowy:** `graf_voronoi_ABCDE.json` – gęsty graf planarny z diagramu Voronoia (20 wierzchołków, 50 krawędzi, rzeczywiste współrzędne 2D)
- **Gramatyka testowa:** `gramatyka_ABCDE.json` – zaawansowana gramatyka siatek trójkątnych z podziałem węzłów (15 produkcji, rozbudowane reguły osadzania)
- Prezentacja strategii selekcji 1 (`ALL_ASCENDING`) oraz 2 (`FIRST_LOWEST`)
- Prezentacja obsługi wiszących krawędzi 1 (`CANCEL`) oraz 2 (`DELETE`)
- Wizualizacje 2D oraz reprezentacja tabelaryczna (HTML i ASCII) połączeń, macierzy sąsiedztwa i krawędzi


In [ ]:
import json
from wptg import (
    Graph,
    Vertex,
    Edge,
    Production,
    Grammar,
    DanglingEdges,
    SelectionStrategy,
    DanglingEdgesHandling,
    TransformationSystem,
    GraphTableFormatter,
)


## 1. Wczytanie i wizualizacja grafu testowego: Diagram Voronoia (`graf_voronoi_ABCDE.json`)
Wczytanie pełnego grafu testowego podanego w specyfikacji zadania. Graf posiada 20 wierzchołków z etykietami od A do E, 50 krawędzi z wagami oraz rzeczywiste współrzędne w przestrzeni 2D.


In [ ]:
with open("graf_voronoi_ABCDE.json", "r", encoding="utf-8") as f:
    voronoi_data = json.load(f)

g_voronoi = Graph.from_dict(voronoi_data)

print(f"Nazwa grafu: {g_voronoi.name}")
print(f"Skierowany: {g_voronoi.directed}")
print(f"Liczba wierzchołków: {len(g_voronoi.vertices)}")
print(f"Liczba krawędzi: {len(g_voronoi.edges)}")


### Wizualizacja grafu Voronoia w przestrzeni 2D
Pakiet `wptg` automatycznie wykorzystuje współrzędne `(x, y)` wierzchołków do odtworzenia geometrii diagramu Voronoia.


In [ ]:
g_voronoi.visualize(figsize=(9, 7))


### Reprezentacja tabelkowa grafu Voronoia
Poniżej przedstawiono tabelę połączeń (stopnie węzłów, etykiety i wagi incydentnych krawędzi) w stylizowanym formacie HTML.


In [ ]:
g_voronoi.display_table(mode="connections")


## 2. Wczytanie oficjalnej gramatyki siatek trójkątnych (`gramatyka_ABCDE.json`)
Gramatyka służy do generowania i zagęszczania siatek trójkątnych. Zawiera 15 produkcji implementujących m.in. doklejanie nowych trójkątów, podział węzła z dziedziczeniem sąsiadów oraz terminację.


In [ ]:
with open("gramatyka_ABCDE.json", "r", encoding="utf-8") as f:
    grammar_data = json.load(f)

gram = Grammar.from_dict(grammar_data)

print(f"Nazwa gramatyki: {gram.name}")
print(f"Nieterminale: {gram.nonterminals}")
print(f"Terminale: {gram.terminals}")
print(f"Domyślna obsługa krawędzi wiszących: {gram.dangling_edges.value}")
print(f"Liczba produkcji: {len(gram.productions)}")


### Zestawienie produkcji gramatyki ABCDE


In [ ]:
for p in gram.productions:
    l_v = len(p.left.vertices)
    r_v = len(p.right.vertices)
    mappings = len(p.mapping)
    embeddings = len(p.embedding)
    print(f"[{p.name.ljust(3)}] {p.description}")
    print(f"      LHS wierzchołki: {l_v}, RHS wierzchołki: {r_v}, mapping: {mappings}, embeddingi: {embeddings}")


### Wizualizacja wybranych nietrywialnych produkcji
Wizualizacja dwudzielna przedstawia lewą stronę (wzorzec) oraz prawą stronę (podstawienie).
Węzły zielone są zachowywane (`mapping`), czerwone są usuwane, a pomarańczowe nowo tworzone.


In [ ]:
p1 = gram.get_production("P1")
p1.visualize(figsize=(10, 4))

p6 = gram.get_production("P6")
p6.visualize(figsize=(10, 4))


## 3. Wyprowadzenie siatki trójkątnej ze startowego grafu gramatyki
Rozpoczynamy od grafu startowego gramatyki (trójkąt A - B - C) i stosujemy sekwencję produkcji:
1. `P1`: doklejenie trójkąta w węźle A ($A \rightarrow A - D - E$)
2. `P6`: podział węzła A ($A \rightarrow B - C$ z regułą osadzenia rozszerzającą krawędzie do obu węzłów)
3. `P5`: doklejenie w węźle E ($E \rightarrow e - B$)
4. `P11`, `P12`, `P13`, `P14`, `P15`: terminacja węzłów (przekształcenie wszystkich nieterminali w małe litery terminalne)


In [ ]:
ts_mesh = TransformationSystem(G=gram.start_graph.copy(), T=gram)

derivation_sequence = ["P1", "P6", "P5", "P11", "P12", "P13", "P14", "P15"]
ts_mesh.apply(derivation_sequence, selection_strategy=2, dangling_edges_handling=1)

for i, (prod, g_step) in enumerate(ts_mesh.History):
    v_str = ", ".join(f"{v.index}:{v.label}" for v in g_step.vertices)
    print(f"Krok {i+1} [{prod.name}]: wierzchołki=[{v_str}], liczba krawędzi={len(g_step.edges)}")


### Wizualizacja ewolucji siatki w kolejnych krokach wyprowadzenia


In [ ]:
ts_mesh.visualize_history(figsize_per_step=(4.2, 3.2))


### Tabele połączeń dla kolejnych kroków wyprowadzenia siatki


In [ ]:
ts_mesh.display_history_tables(mode="connections")


## 4. Nietrywialne transformacje na pełnym grafie Voronoia
Demonstracja obu wymaganych strategii selekcji na rzeczywistym grafie Voronoia o 20 wierzchołkach i 50 krawędziach.

### Strategia 2 (`FIRST_LOWEST`): Wybór pasującego węzła o najniższym indeksie
Aplikujemy produkcję `P6` ($A \rightarrow B - C$), która usuwa wierzchołek A i zastępuje go parą B i C połączoną nową krawędzią.
Węzeł 0 (A) ma stopień 7 w grafie Voronoia. Zgodnie z regułą osadzania `neighborLabel: "*", connectTo: [0, 1]`, wszystkie 7 krawędzi zostaje zduplikowanych do obu nowych wierzchołków.


In [ ]:
ts_voronoi_s2 = TransformationSystem(G=g_voronoi.copy(), T=gram)

print(f"Początkowa liczba wierzchołków: {len(ts_voronoi_s2.G.vertices)}")
print(f"Początkowa liczba krawędzi: {len(ts_voronoi_s2.G.edges)}")

applied_p6 = ts_voronoi_s2.apply(["P6"], selection_strategy=2, dangling_edges_handling=1)

print(f"Zastosowano kroków: {len(applied_p6)}")
print(f"Liczba wierzchołków po P6: {len(ts_voronoi_s2.G.vertices)}")
print(f"Liczba krawędzi po P6: {len(ts_voronoi_s2.G.edges)}")

ts_voronoi_s2.G.display_table(mode="connections")


### Strategia 1 (`ALL_ASCENDING`): Zastosowanie do wszystkich pasujących węzłów rosnąco
Stosujemy produkcję terminującą `P11` ($A \rightarrow a$) ze strategią 1.
Wszystkie wierzchołki o etykiecie A w całym grafie Voronoia zostają w jednym kroku przekształcone w terminale a w kolejności rosnących indeksów.


In [ ]:
ts_voronoi_s1 = TransformationSystem(G=ts_voronoi_s2.G.copy(), T=gram)

applied_p11 = ts_voronoi_s1.apply(["P11"], selection_strategy=1, dangling_edges_handling=1)

print(f"Liczba wykonanych przekształceń w kroku P11: {len(applied_p11)}")
for prod, g_step in applied_p11:
    v_a = [v.index for v in g_step.vertices if v.label == "a"]
    print(f"Zastosowano {prod.name}, zaktualizowane terminale 'a': {v_a}")

ts_voronoi_s1.G.display_table(mode="connections")


## 5. Obsługa wiszących krawędzi: Anulowanie (`CANCEL` = 1) vs Usunięcie (`DELETE` = 2)
W tej sekcji sprawdzamy zachowanie systemu, gdy usuwany jest wierzchołek posiadający krawędzie do reszty grafu, dla których nie zdefiniowano reguły osadzenia (embedding).
Definiujemy produkcję `P_delete_C`, która usuwa węzeł C bez reguł embeddingu, i testujemy ją na węźle 1 w grafie Voronoia (węzeł 1 to C posiadający wielu sąsiadów).


In [ ]:
P_delete_C = Production(
    name="P_delete_C_no_embedding",
    left=Graph(vertices=[Vertex(0, "C")]),
    right=Graph(vertices=[Vertex(0, "c_isolated")]),
    mapping=[],
    embedding=[]
)

print(f"Węzeł 1 w grafie Voronoia: {g_voronoi.get_vertex(1)}")
incident_to_1 = [e for e in g_voronoi.edges if e.u == 1 or e.v == 1]
print(f"Liczba krawędzi incydentnych z węzłem 1: {len(incident_to_1)}")

ts_cancel = TransformationSystem(G=g_voronoi.copy(), T=gram)
res_cancel = ts_cancel.apply([P_delete_C], selection_strategy=2, dangling_edges_handling=1)
print(f"Tryb 1 (CANCEL): liczba wykonanych transformacji = {len(res_cancel)} (brak modyfikacji grafu)")

ts_delete = TransformationSystem(G=g_voronoi.copy(), T=gram)
res_delete = ts_delete.apply([P_delete_C], selection_strategy=2, dangling_edges_handling=2)
print(f"Tryb 2 (DELETE): liczba wykonanych transformacji = {len(res_delete)}")
print(f"Węzeł 1 obecny po DELETE: {ts_delete.G.get_vertex(1) is not None}")
print(f"Liczba krawędzi przed: {len(g_voronoi.edges)}, po DELETE: {len(ts_delete.G.edges)}")


## 6. Zestawienie reprezentacji tabelarycznych grafu
Pakiet `wptg` udostępnia reprezentacje w postaci:
- Tabeli połączeń (stopnie wierzchołków, etykiety i wagi)
- Macierzy sąsiedztwa
- Listy krawędzi
Zarówno w wersji HTML (dla Jupyter Notebook) jak i w czystej wersji tekstowej ASCII (do logów i konsoli).


In [ ]:
small_sample_graph = ts_mesh.G

print("=== Reprezentacja tekstowa ASCII: Tabela połączeń ===")
small_sample_graph.print_table(mode="connections")

print("=== Reprezentacja tekstowa ASCII: Macierz sąsiedztwa ===")
small_sample_graph.print_table(mode="matrix")

print("=== Reprezentacja tekstowa ASCII: Lista krawędzi ===")
small_sample_graph.print_table(mode="edges")

print("=== Reprezentacja HTML (macierz sąsiedztwa) ===")
small_sample_graph.display_table(mode="matrix")
